# Prepare the datasets

Validation, standardization, deduplication, Morgan fingerprints and clustering for the p38α,
COX-2 and 5-LOX datasets. Output goes to `analysis/{slug}/prepared/`.

Structures are handled with RDKit's `rdMolStandardize`, which holds the MolVS algorithms.
Labels from notebook 1 are carried through unchanged.

In [ ]:
import json
import platform
import sys
import time
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rdkit
from IPython.display import Markdown, display
from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import Descriptors, Draw, rdFingerprintGenerator
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit.SimDivFilters import rdSimDivPickers

FP_RADIUS = 2
FP_BITS = 2048
FP_USE_CHIRALITY = False
DEDUPE_ON_STEREO = True
CANONICAL_TAUTOMER = False
CLUSTER_DISTANCE = 0.35
RANDOM_SEED = 0

TARGETS = {
    'p38':  {'chembl_id': 'CHEMBL260', 'name': 'p38alpha (MAPK14)'},
    'cox2': {'chembl_id': 'CHEMBL230', 'name': 'COX-2 (PTGS2)'},
    'lox5': {'chembl_id': 'CHEMBL215', 'name': '5-LOX (ALOX5)'},
}

import os
if os.environ.get('NL_SLUGS'):
    TARGETS = {k: v for k, v in TARGETS.items() if k in os.environ['NL_SLUGS'].split(',')}

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'data/p38_training_raw.csv').exists()), None)
if ROOT is None:
    raise RuntimeError('No data/p38_training_raw.csv found. Start Jupyter from the '
                       'NeuroLead project or its src directory, and run notebook 1 first.')

RDLogger.DisableLog('rdApp.*')
np.random.seed(RANDOM_SEED)
plt.rcParams.update({'figure.dpi': 110, 'font.size': 9, 'axes.grid': True,
                     'grid.alpha': 0.3, 'axes.axisbelow': True})


def banner(slug):
    t = TARGETS[slug]
    display(Markdown(f'#### {t["name"]} &middot; `{t["chembl_id"]}` &middot; slug `{slug}`'))


print(f'python {sys.version.split()[0]} | rdkit {rdkit.__version__} | pandas {pd.__version__}')
S = {}
for slug in TARGETS:
    source = ROOT / f'data/{slug}_training_raw.csv'
    out = ROOT / f'analysis/{slug}/prepared'
    out.mkdir(parents=True, exist_ok=True)
    raw = pd.read_csv(source)
    S[slug] = {'source': source, 'out': out, 'raw': raw,
               'cascade': [('rows in source CSV', len(raw))]}
    banner(slug)
    print(f'{source.relative_to(ROOT)}: {len(raw):,} rows x {raw.shape[1]} columns')
    print(f'labels: {dict(raw.label.value_counts())}')
    display(raw.head(3))

## 2. Validate structures

In [ ]:
validator = rdMolStandardize.MolVSValidation()
METALS = {'Li', 'Na', 'K', 'Mg', 'Ca', 'Fe', 'Zn', 'Cu', 'Mn', 'Pt', 'Co', 'Ni', 'Al', 'Ag', 'Au',
          'Cd', 'Hg', 'Sn', 'Bi', 'Ba', 'Sr', 'Cr', 'Mo', 'Pd', 'Ru'}


def validate(raw):
    records = []
    for row in raw.itertuples():
        mol = Chem.MolFromSmiles(row.canonical_smiles)
        if mol is None:
            records.append({'parses': False, 'n_fragments': 0, 'messages': 'SMILES failed to parse'})
            continue
        messages = [m.split('] ', 1)[-1] for m in validator.validate(mol)]
        records.append({
            'parses': True,
            'n_fragments': len(Chem.GetMolFrags(mol)),
            'has_metal': any(a.GetSymbol() in METALS for a in mol.GetAtoms()),
            'has_isotope': any(a.GetIsotope() for a in mol.GetAtoms()),
            'has_radical': any(a.GetNumRadicalElectrons() for a in mol.GetAtoms()),
            'net_charge': Chem.GetFormalCharge(mol),
            'messages': ' | '.join(messages),
        })
    return pd.DataFrame(records, index=raw.index)


for slug, s in S.items():
    banner(slug)
    t0 = time.time()
    validation = validate(s['raw'])
    s['validation'] = validation
    print(f'validated {len(validation):,} structures in {time.time() - t0:.1f}s\n')

    print(f'failed to parse        {int((~validation.parses).sum()):>5}')
    print(f'multiple fragments     {int((validation.n_fragments > 1).sum()):>5}   (salts, mixtures)')
    print(f'contains a metal       {int(validation.has_metal.fillna(False).sum()):>5}')
    print(f'isotope-labelled       {int(validation.has_isotope.fillna(False).sum()):>5}')
    print(f'has radical electrons  {int(validation.has_radical.fillna(False).sum()):>5}')
    print(f'net charge != 0        {int(validation.net_charge.fillna(0).ne(0).sum()):>5}')

    issues = Counter(m for msg in validation.messages.dropna() if msg for m in msg.split(' | '))
    print('\nMolVS validation messages:')
    for message, n in issues.most_common():
        print(f'  {n:>5}  {message}')
    if not issues:
        print('  none')

## 3. Standardize

Cleanup, largest-fragment parent and neutralisation. Canonical tautomers are available but off
by default.

In [ ]:
uncharger = rdMolStandardize.Uncharger()
tautomer_enumerator = rdMolStandardize.TautomerEnumerator()


def standardize(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None, None, 'unparseable'
    before = Chem.MolToSmiles(mol)
    changes = []
    try:
        mol = rdMolStandardize.Cleanup(mol)
        if len(Chem.GetMolFrags(mol)) > 1:
            mol = rdMolStandardize.FragmentParent(mol)
            changes.append('fragment_removed')
        if Chem.GetFormalCharge(mol) != 0:
            mol = uncharger.uncharge(mol)
            changes.append('neutralized')
        if CANONICAL_TAUTOMER:
            mol = tautomer_enumerator.Canonicalize(mol)
            changes.append('tautomer_canonicalized')
        Chem.SanitizeMol(mol)
    except Exception as exc:
        return None, None, f'standardization failed: {type(exc).__name__}'
    after = Chem.MolToSmiles(mol)
    if after != before and 'fragment_removed' not in changes and 'neutralized' not in changes:
        changes.append('normalized')
    flat = Chem.MolToSmiles(mol, isomericSmiles=False)
    return after, flat, ','.join(changes)


for slug, s in S.items():
    banner(slug)
    raw = s['raw']
    t0 = time.time()
    std = pd.DataFrame([standardize(sm) for sm in raw.canonical_smiles],
                       columns=['std_smiles', 'std_smiles_flat', 'std_changes'], index=raw.index)
    print(f'standardized {len(std):,} molecules in {time.time() - t0:.0f}s'
          f"{' (incl. tautomer canonicalization)' if CANONICAL_TAUTOMER else ''}\n")

    failed = std.std_smiles.isna()
    print(f'{int(failed.sum())} molecules could not be standardized')
    if failed.any():
        display(pd.concat([raw.loc[failed, ['molecule_chembl_id', 'label']],
                           std.loc[failed, ['std_changes']]], axis=1))

    print(f'\n{int(std.std_changes.ne("").sum()):,} molecules were altered:')
    for change, n in Counter(c for c in std.std_changes if c).most_common():
        print(f'  {n:>5}  {change}')

    mols = pd.concat([raw, s['validation'].drop(columns='messages'), std], axis=1)
    mols = mols.loc[~failed].reset_index(drop=True)
    s['cascade'].append(('standardized successfully', len(mols)))
    s['mols'] = mols

## 4. Duplicates

Duplicates are found after standardization. When several entries share a structure, the one
with the most measurements is kept.

In [ ]:
key = 'std_smiles' if DEDUPE_ON_STEREO else 'std_smiles_flat'
print(f'deduplicating on {key} (stereochemistry '
      f'{"distinguishes" if DEDUPE_ON_STEREO else "does not distinguish"} molecules)')

for slug, s in S.items():
    banner(slug)
    mols = s['mols']
    groups = mols.groupby(key)
    mols['dup_group_size'] = groups[key].transform('size')
    mols['dup_label_conflict'] = groups.label.transform('nunique') > 1

    dups = mols.loc[mols.dup_group_size > 1].sort_values([key, 'n_measurements'])
    print(f'{int(mols.dup_group_size.gt(1).sum()):,} rows fall into '
          f'{dups[key].nunique():,} duplicated structures')
    print(f'{int(mols.dup_label_conflict.sum()):,} of those rows disagree on the label')

    flat_extra = mols.std_smiles_flat.duplicated().sum() - mols.std_smiles.duplicated().sum()
    print(f'ignoring stereochemistry would merge {int(flat_extra):,} further rows')

    if len(dups):
        display(dups[['molecule_chembl_id', 'label', 'pActivity', 'pActivity_min', 'pActivity_max',
                      'n_measurements', 'std_changes', 'dup_group_size', 'dup_label_conflict']])

    clean = (mols.sort_values(['n_measurements', 'molecule_chembl_id'], ascending=[False, True])
                 .drop_duplicates(subset=key, keep='first')
                 .sort_values('pActivity', ascending=False)
                 .reset_index(drop=True))
    s['cascade'].append(('unique standardized structures', len(clean)))
    s['clean'] = clean
    print(f'\n{len(clean):,} unique structures retained')

## 5. Cleaning cascade

In [ ]:
for slug, s in S.items():
    banner(slug)
    raw, clean = s['raw'], s['clean']
    cascade_df = pd.DataFrame(s['cascade'], columns=['step', 'rows'])
    cascade_df['removed'] = (-cascade_df.rows.diff()).fillna(0).astype(int)
    cascade_df['kept_pct'] = (cascade_df.rows / len(raw) * 100).round(2)
    display(cascade_df)

    print('label composition through cleaning')
    display(pd.DataFrame({'source': raw.label.value_counts(),
                          'cleaned': clean.label.value_counts()}).fillna(0).astype(int))

    binary = clean.loc[clean.label.ne('dropped')]
    ratio = (binary.label.eq("active").sum() / max(binary.label.eq("inactive").sum(), 1))
    print(f'\nbinary modelling set: {len(binary):,} molecules '
          f'({int(binary.label.eq("active").sum()):,} active / '
          f'{int(binary.label.eq("inactive").sum()):,} inactive, {ratio:.0f}:1)')

## 6. Potency distributions

In [ ]:
colors = {'active': '#2d6a4f', 'inactive': '#9d2235', 'dropped': '#9a9a9a'}

for slug, s in S.items():
    banner(slug)
    clean = s['clean']
    fig, axes = plt.subplots(2, 2, figsize=(11, 7.5))

    ax = axes[0, 0]
    ax.hist(clean.pActivity, bins=60, color='#4a6fa5', edgecolor='white', linewidth=0.3)
    for x, text in [(5.0, 'inactive < 5.0'), (6.0, 'active >= 6.0')]:
        ax.axvline(x, color='#c1121f', linestyle='--', linewidth=1)
        ax.text(x, ax.get_ylim()[1] * 0.95, f' {text}', fontsize=7.5, color='#c1121f')
    ax.axvspan(5.0, 6.0, color='#c1121f', alpha=0.06)
    ax.set(title='pActivity distribution, all molecules', xlabel='pActivity', ylabel='molecules')

    ax = axes[0, 1]
    for label in ['active', 'dropped', 'inactive']:
        values = clean.loc[clean.label.eq(label), 'pActivity']
        if len(values) > 1:
            values.plot.kde(ax=ax, color=colors[label], linewidth=1.6,
                            label=f'{label} (n={len(values):,})')
    ax.set(title='Density by label', xlabel='pActivity', xlim=(3, 12))
    ax.legend(fontsize=7.5)

    ax = axes[1, 0]
    order = clean.standard_type.value_counts().index.tolist()
    ax.boxplot([clean.loc[clean.standard_type.eq(t), 'pActivity'] for t in order],
               tick_labels=order, vert=True, widths=0.6, patch_artist=True,
               boxprops={'facecolor': '#cfd8e3', 'linewidth': 0.8},
               medianprops={'color': '#c1121f', 'linewidth': 1.4}, flierprops={'markersize': 2})
    ax.axhline(6.0, color='#c1121f', linestyle='--', linewidth=0.9)
    ax.set(title='Endpoint type vs potency', ylabel='pActivity')

    ax = axes[1, 1]
    replicated = clean.n_measurements.gt(1)
    ax.hist([clean.loc[~replicated, 'pActivity'], clean.loc[replicated, 'pActivity']],
            bins=40, stacked=True, color=['#b8c4d4', '#2d6a4f'], edgecolor='white', linewidth=0.3,
            label=[f'single measurement (n={int((~replicated).sum()):,})',
                   f'replicated (n={int(replicated.sum()):,})'])
    ax.set(title='Replication across the potency range', xlabel='pActivity', ylabel='molecules')
    ax.legend(fontsize=7.5)

    fig.suptitle(f'{slug} potency after cleaning', fontsize=11, y=0.995)
    fig.tight_layout()
    fig.savefig(s['out'] / 'potency_distributions.png', dpi=150, bbox_inches='tight')
    plt.show()

    display(clean.groupby('label').pActivity.describe().round(2))

## 7. Morgan fingerprints

Radius 2, 2048 bits, chirality ignored, computed from the standardized SMILES. Notebook 7 uses
the same settings for the plant compounds.

In [ ]:
generator = rdFingerprintGenerator.GetMorganGenerator(
    radius=FP_RADIUS, fpSize=FP_BITS, includeChirality=FP_USE_CHIRALITY)

for slug, s in S.items():
    banner(slug)
    clean, out = s['clean'], s['out']
    t0 = time.time()
    fp_bits, fp_objects = [], []
    for smiles in clean.std_smiles:
        mol = Chem.MolFromSmiles(smiles)
        fp = generator.GetFingerprint(mol)
        fp_objects.append(fp)
        fp_bits.append(np.frombuffer(bytes(fp.ToBitString(), 'ascii'), 'u1') - ord('0'))

    X_bits = np.vstack(fp_bits).astype(np.uint8)
    print(f'{X_bits.shape[0]:,} x {X_bits.shape[1]} fingerprints in {time.time() - t0:.0f}s')

    density = X_bits.mean()
    on_per_mol = X_bits.sum(axis=1)
    never_set = int((X_bits.sum(axis=0) == 0).sum())
    print(f'\nmatrix density          {density:.3%}')
    print(f'bits set per molecule   median {int(np.median(on_per_mol))}, '
          f'range {on_per_mol.min()}-{on_per_mol.max()}')
    print(f'bits never set          {never_set:,} of {FP_BITS} '
          f'({never_set / FP_BITS:.0%} carry no information for this dataset)')
    print(f'all-zero fingerprints   {int((on_per_mol == 0).sum())}')

    np.save(out / 'morgan_bits.npy', X_bits)
    clean[['molecule_chembl_id']].to_csv(out / 'fingerprint_row_order.csv', index=False)
    print(f'\nsaved to {out.relative_to(ROOT).as_posix()}/ — row order matches fingerprint_row_order.csv')
    s.update(X_bits=X_bits, fp_objects=fp_objects, density=density, never_set=never_set)

### Fingerprint collisions

Different molecules that share an identical fingerprint.

In [ ]:
for slug, s in S.items():
    banner(slug)
    clean, X_bits = s['clean'], s['X_bits']
    packed = np.packbits(X_bits, axis=1)
    _, inverse, counts = np.unique(packed, axis=0, return_inverse=True, return_counts=True)
    inverse = inverse.ravel()
    collided = np.where(counts > 1)[0]
    s['collided'] = collided
    print(f'{len(collided):,} fingerprints are shared by more than one molecule '
          f'({int(counts[collided].sum()):,} molecules involved)')

    if len(collided):
        rows = []
        for group in collided:
            members = clean.iloc[np.where(inverse == group)[0]]
            rows.append({'molecules': ', '.join(members.molecule_chembl_id),
                         'labels': ', '.join(members.label),
                         'pActivity': ', '.join(f'{v:.2f}' for v in members.pActivity),
                         'label_conflict': members.label.nunique() > 1})
        collisions = pd.DataFrame(rows).sort_values('label_conflict', ascending=False)
        display(collisions.head(15))
        collisions.to_csv(s['out'] / 'fingerprint_collisions.csv', index=False)
        n_conflict = int(collisions.label_conflict.sum())
        print(f'\n{n_conflict} collision groups have conflicting labels')

## 8. Clustering

Leader-picker clustering at Tanimoto similarity ≥ 0.65. Each molecule is assigned to its
nearest cluster centre.

In [ ]:
for slug, s in S.items():
    banner(slug)
    clean, fp_objects = s['clean'], s['fp_objects']
    t0 = time.time()
    picker = rdSimDivPickers.LeaderPicker()
    leader_idx = list(picker.LazyBitVectorPick(fp_objects, len(fp_objects), CLUSTER_DISTANCE))
    leaders = [fp_objects[i] for i in leader_idx]

    assignment, nearest = [], []
    for fp in fp_objects:
        sims = np.array(DataStructs.BulkTanimotoSimilarity(fp, leaders))
        best = int(sims.argmax())
        assignment.append(best)
        nearest.append(float(sims[best]))

    clean['cluster'] = assignment
    clean['similarity_to_centre'] = np.round(nearest, 3)
    sizes = clean.cluster.value_counts()

    print(f'{len(leaders):,} clusters at Tanimoto similarity >= {1 - CLUSTER_DISTANCE:.2f} '
          f'({time.time() - t0:.0f}s)')
    print(f'singletons              {int((sizes == 1).sum()):,} '
          f'({(sizes == 1).sum() / len(sizes):.0%} of clusters)')
    print(f'largest cluster         {int(sizes.iloc[0]):,} molecules')
    print(f'top 10 clusters hold    {int(sizes.head(10).sum()):,} molecules '
          f'({sizes.head(10).sum() / len(clean):.0%} of the dataset)')

    summary = clean.groupby('cluster').agg(
        molecules=('molecule_chembl_id', 'size'),
        active=('label', lambda x: int(x.eq('active').sum())),
        inactive=('label', lambda x: int(x.eq('inactive').sum())),
        dropped=('label', lambda x: int(x.eq('dropped').sum())),
        median_pActivity=('pActivity', 'median'),
        pActivity_range=('pActivity', lambda x: round(x.max() - x.min(), 2)),
    ).sort_values('molecules', ascending=False)
    summary.insert(0, 'centre', [clean.std_smiles.iloc[leader_idx[c]] for c in summary.index])
    summary.to_csv(s['out'] / 'cluster_summary.csv')
    display(summary.head(12))
    s.update(leaders=leaders, leader_idx=leader_idx, sizes=sizes, summary=summary)

In [ ]:
for slug, s in S.items():
    banner(slug)
    sizes, summary = s['sizes'], s['summary']
    fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))

    ax = axes[0]
    ax.hist(sizes.values, bins=np.arange(1, min(sizes.max(), 60) + 2) - 0.5,
            color='#4a6fa5', edgecolor='white', linewidth=0.3)
    ax.set(title='Cluster size distribution', xlabel='molecules in cluster',
           ylabel='clusters', yscale='log')

    ax = axes[1]
    top = summary.head(15).iloc[::-1]
    positions = np.arange(len(top))
    for column, color in [('active', '#2d6a4f'), ('dropped', '#9a9a9a'), ('inactive', '#9d2235')]:
        left = top[['active', 'dropped']].iloc[:, :['active', 'dropped', 'inactive'].index(column)].sum(axis=1)
        ax.barh(positions, top[column], left=left, color=color, height=0.75, label=column)
    ax.set(title='Label mix, 15 largest clusters', xlabel='molecules',
           yticks=positions, yticklabels=[f'c{i}' for i in top.index])
    ax.tick_params(labelsize=7)
    ax.legend(fontsize=7.5)

    ax = axes[2]
    inactive_clusters = int((summary.inactive > 0).sum())
    ax.scatter(summary.molecules, summary.median_pActivity, s=14, alpha=0.45,
               c=np.where(summary.inactive > 0, '#9d2235', '#4a6fa5'), linewidths=0)
    ax.axhline(6.0, color='#c1121f', linestyle='--', linewidth=0.9)
    ax.set(title=f'{inactive_clusters} clusters contain an inactive', xlabel='cluster size',
           ylabel='median pActivity', xscale='log')

    fig.suptitle(f'{slug} clustering', fontsize=11)
    fig.tight_layout()
    fig.savefig(s['out'] / 'clustering.png', dpi=150, bbox_inches='tight')
    plt.show()

    print(f'Only {inactive_clusters:,} of {len(summary):,} clusters contain any inactive molecule.')

## 9. Export

In [ ]:
columns = ['molecule_chembl_id', 'molecule_pref_name', 'canonical_smiles', 'std_smiles',
           'std_smiles_flat', 'std_changes', 'label', 'pActivity', 'pActivity_min',
           'pActivity_max', 'n_measurements', 'standard_type', 'confidence_score',
           'n_fragments', 'net_charge', 'dup_group_size', 'dup_label_conflict',
           'cluster', 'similarity_to_centre']

for slug, s in S.items():
    banner(slug)
    clean, out, X_bits, sizes, summary = s['clean'], s['out'], s['X_bits'], s['sizes'], s['summary']
    prepared = clean[columns]
    prepared.to_csv(out / f'{slug}_prepared.csv', index=False, encoding='utf-8')
    s['prepared'] = prepared

    manifest = {
        'generated': pd.Timestamp.now().isoformat(timespec='seconds'),
        'target_chembl_id': TARGETS[slug]['chembl_id'],
        'target_name': TARGETS[slug]['name'],
        'source_csv': str(s['source'].relative_to(ROOT)),
        'source_rows': len(s['raw']),
        'prepared_rows': len(prepared),
        'cascade': {step: int(n) for step, n in s['cascade']},
        'labels': {k: int(v) for k, v in prepared.label.value_counts().items()},
        'settings': {
            'fp_radius': FP_RADIUS, 'fp_bits': FP_BITS, 'fp_use_chirality': FP_USE_CHIRALITY,
            'dedupe_on_stereo': DEDUPE_ON_STEREO, 'canonical_tautomer': CANONICAL_TAUTOMER,
            'cluster_distance': CLUSTER_DISTANCE, 'random_seed': RANDOM_SEED,
        },
        'standardization_changes': {k: int(v) for k, v in
                                    Counter(c for c in clean.std_changes if c).items()},
        'clusters': {'n_clusters': len(s['leaders']), 'singletons': int((sizes == 1).sum()),
                     'largest': int(sizes.iloc[0]),
                     'clusters_with_inactives': int((summary.inactive > 0).sum())},
        'fingerprint_matrix': {'shape': list(X_bits.shape), 'density': round(float(s['density']), 5),
                               'bits_never_set': s['never_set'],
                               'collision_groups': int(len(s['collided']))},
        'versions': {'python': sys.version.split()[0], 'platform': platform.platform(),
                     'rdkit': rdkit.__version__, 'pandas': pd.__version__, 'numpy': np.__version__},
        'note': ('Labels and cutoffs are carried through unchanged from src/1_fetch_p38.ipynb. '
                 'The source CSV was not modified.'),
    }
    s['manifest'] = manifest
    (out / 'prepare_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')

    for path in sorted(out.iterdir()):
        print(f'  {path.name:<42} {path.stat().st_size / 1024:>9,.1f} KB')
    print(f'in {out.relative_to(ROOT).as_posix()}/')

## 10. The three targets side by side

The numbers from the manifests in one table. `7_plant_similarity.ipynb` uses the `active`
rows of each prepared table as its reference set.

In [ ]:
rows = {}
for slug, s in S.items():
    m, p = s['manifest'], s['prepared']
    rows[slug] = {'target': m['target_chembl_id'],
                  **{f'cascade: {k}': v for k, v in m['cascade'].items()},
                  **{f'label: {k}': m['labels'].get(k, 0) for k in ['active', 'dropped', 'inactive']},
                  'actives used by notebook 7 (pActivity >= 6)': int((p.pActivity >= 6).sum()),
                  'clusters': m['clusters']['n_clusters'],
                  'singleton clusters': m['clusters']['singletons'],
                  'largest cluster': m['clusters']['largest'],
                  'clusters with an inactive': m['clusters']['clusters_with_inactives'],
                  'median bits set per molecule': int(np.median(s['X_bits'].sum(axis=1))),
                  'fingerprint collision groups': m['fingerprint_matrix']['collision_groups']}
display(pd.DataFrame(rows))